# Image Resizer — resize + contrast, under a size cap
---
Resizes images (by percentage), optionally boosts contrast, and re-saves each one under a
maximum file size. Storage and input are now flexible, the same way as the Seaweed pipeline.

**Where files live (`STORAGE_BACKEND`)**
- `"drive"` — Google Drive (one-click permission popup)
- `"onedrive"` — Microsoft OneDrive (one-time Azure app setup, see the *Connect* cell)
- `"none"` — no cloud storage at all: upload your files, download a zip of the results

**What you can give it (auto-detected)**
- a `.zip` of images (subfolders are fine)
- a `.pdf` (each page becomes one image)
- a folder of loose images (`.jpg` / `.jpeg` / `.png` / `.webp`)
- a single image file
- or, with `UPLOAD_INSTEAD = True`, pick any mix of the above from a file-picker widget

### How to use this notebook
Everything you might want to change is in **one configuration cell** right below. Set it up,
then `Runtime > Run all`. You shouldn't need to edit anything after that cell.

## Configuration
---

In [1]:
# =============================================================================
# STORAGE — where results are saved (and where source files can be read from)
# =============================================================================
STORAGE_BACKEND = "none"   # "drive", "onedrive", or "none"
                             #   "none" = no cloud storage: you MUST use UPLOAD_INSTEAD = True,
                             #   and the results are downloaded to your computer as a zip.

ONEDRIVE_CLIENT_ID = ""     # <-- only needed if STORAGE_BACKEND = "onedrive"
                             #     (Azure app registration — see the "Connect" cell below)

# =============================================================================
# INPUT SOURCE — where your source material is coming from
# =============================================================================
# Set UPLOAD_INSTEAD = True to skip the path settings below and get a file-picker widget when
# you run the "Get your images" cell. Select a zip, a PDF, one image, or several images at once.
UPLOAD_INSTEAD = True

# If UPLOAD_INSTEAD is False, point at your source. All of these work, auto-detected from the
# path/extension: a .zip of images | a .pdf | a folder of loose images | a single image file
SOURCE_PATH = "/content/drive/MyDrive/S-260706-142934.zip"
# ^ used when STORAGE_BACKEND = "drive"

ONEDRIVE_SOURCE_PATH = "/PipelineInput/S-260706-142934.zip"
# ^ used when STORAGE_BACKEND = "onedrive" — path inside your OneDrive

PDF_DPI = 300   # only used if your source is a PDF — resolution pages are rasterised at

# =============================================================================
# RESIZE SETTINGS
# =============================================================================
resize_percent = 100        # e.g. 50 = half the original size (100 is full)
max_size_bytes = 10 * 1024 * 1024  # 10 MB cap per output file
enhance_contrast = True     # set False to skip contrast enhancement entirely
contrast_factor = 1.4       # >1.0 boosts contrast, 1.0 = no change. Try 1.2–1.6 for faint pencil/pale ink

# =============================================================================
# OUTPUT
# =============================================================================
save_name = "AntonioPiaggio_prepped_photos"  # <-- results are saved under this name
DOWNLOAD_ZIP = False   # True = ALSO download a zip of the results to your computer when done
                        # (always happens when STORAGE_BACKEND = "none")

## Setup — connect to storage
---
Uses the `STORAGE_BACKEND` you set above.

**Google Drive** connects with one click (a permissions popup) — no setup required.

**OneDrive** needs a one-time setup because Colab has no native OneDrive mount, so this
notebook talks to it through Microsoft's Graph API instead:
1. Go to [portal.azure.com](https://portal.azure.com) > **App registrations** > **New registration**
2. Name it anything (e.g. "Colab Pipeline"), leave redirect URI blank, register
3. On the app's **Authentication** page, set **"Allow public client flows"** to **Yes**
4. On **API permissions**, add Microsoft Graph > Delegated > `Files.ReadWrite.All`, then `Add permissions`
5. Copy the **Application (client) ID** from the app's Overview page into `ONEDRIVE_CLIENT_ID` above

You only do this once — the same client ID works for every future session. Running this cell
then prints a short code and a link: visit the link on any device, enter the code, sign in.

**"none"** skips this step entirely.

In [2]:
import os

onedrive_access_token = None
GRAPH_BASE = "https://graph.microsoft.com/v1.0/me/drive/root:"

if STORAGE_BACKEND == "none" and not UPLOAD_INSTEAD:
    raise ValueError('STORAGE_BACKEND = "none" has nowhere to read files from — set UPLOAD_INSTEAD = True.')

if STORAGE_BACKEND == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    print("Connected to Google Drive.")

elif STORAGE_BACKEND == "onedrive":
    !pip install -q msal requests
    import msal, requests

    if not ONEDRIVE_CLIENT_ID:
        raise ValueError("Set ONEDRIVE_CLIENT_ID in the config cell above first — see setup instructions.")

    GRAPH_SCOPES = ["Files.ReadWrite.All", "offline_access"]
    app = msal.PublicClientApplication(
        ONEDRIVE_CLIENT_ID,
        authority="https://login.microsoftonline.com/consumers"
    )

    flow = app.initiate_device_flow(scopes=GRAPH_SCOPES)
    if "user_code" not in flow:
        raise Exception(f"Device flow failed to start: {flow}")
    print(flow["message"])  # "To sign in, visit https://microsoft.com/devicelogin and enter code XXXXX"

    token_result = app.acquire_token_by_device_flow(flow)  # blocks until you complete sign-in

    if "access_token" not in token_result:
        raise Exception(f"Auth failed: {token_result.get('error_description')}")

    onedrive_access_token = token_result["access_token"]
    print("Connected to OneDrive.")

elif STORAGE_BACKEND == "none":
    print("No cloud storage selected — using upload + download.")

else:
    raise ValueError('STORAGE_BACKEND must be "drive", "onedrive", or "none"')


def _graph_headers():
    return {"Authorization": f"Bearer {onedrive_access_token}"}

def onedrive_download_file(remote_path, local_path):
    import requests
    url = f"{GRAPH_BASE}{remote_path}:/content"
    r = requests.get(url, headers=_graph_headers())
    r.raise_for_status()
    os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
    with open(local_path, "wb") as f:
        f.write(r.content)

def onedrive_upload_folder(local_dir, remote_folder):
    import requests
    for fname in sorted(os.listdir(local_dir)):
        local_path = os.path.join(local_dir, fname)
        if not os.path.isfile(local_path):
            continue
        url = f"{GRAPH_BASE}{remote_folder}/{fname}:/content"
        with open(local_path, "rb") as f:
            r = requests.put(url, headers=_graph_headers(), data=f)
        if r.status_code not in (200, 201):
            print(f"  upload failed for {fname}: {r.status_code} {r.text[:200]}")

def onedrive_download_folder(remote_folder, local_dir):
    import requests
    os.makedirs(local_dir, exist_ok=True)
    url = f"{GRAPH_BASE}{remote_folder}:/children"
    r = requests.get(url, headers=_graph_headers())
    r.raise_for_status()
    for item in r.json().get("value", []):
        if "file" in item:
            resp = requests.get(item["@microsoft.graph.downloadUrl"])
            with open(os.path.join(local_dir, item["name"]), "wb") as f:
                f.write(resp.content)

def save_to_storage(local_dir, name):
    """Persist a local folder to whichever backend is selected."""
    import shutil
    if STORAGE_BACKEND == "drive":
        dest = f"/content/drive/MyDrive/{name}"
        shutil.copytree(local_dir, dest, dirs_exist_ok=True)
        print(f"Saved to Google Drive at: {dest}")
        return dest
    else:
        remote_folder = f"/{name}"
        onedrive_upload_folder(local_dir, remote_folder)
        print(f"Uploaded to OneDrive at: {remote_folder}")
        return remote_folder

No cloud storage selected — using upload + download.


## Step 1 — Get your source images
---
Reads `UPLOAD_INSTEAD` / `SOURCE_PATH` / `ONEDRIVE_SOURCE_PATH` from the config cell.

- **Upload widget** (`UPLOAD_INSTEAD = True`): a file picker appears when you run this cell.
  Select a zip, a PDF, one image, or several images — any mix is fine.
- **Drive / OneDrive path**: point at a `.zip`, a `.pdf`, a single image, or a folder of loose images.

Either way, by the end of this cell every source page is a plain image file in `input_dir`.
Zips can contain subfolders; macOS `__MACOSX` / `._` junk files are ignored. This cell also
clears `input_dir` and `output_dir` first, so re-running it never mixes in leftovers from a
previous run.

In [3]:
import zipfile
import shutil
import tempfile

!pip install -q pymupdf
import fitz  # PyMuPDF, for PDF -> image conversion

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")

input_dir = "/content/input_images"
output_dir = "/content/output_images"
for d in (input_dir, output_dir):
    shutil.rmtree(d, ignore_errors=True)   # start clean on every run
    os.makedirs(d)


def is_image(fname):
    base = os.path.basename(fname)
    return base.lower().endswith(IMAGE_EXTS) and not base.startswith(".")  # skip macOS ._ files

def _copy_into_input(src_path, name):
    """Copy into input_dir, renaming on collision (so flattened subfolders don't overwrite)."""
    stem, ext = os.path.splitext(name)
    dest, k = os.path.join(input_dir, name), 1
    while os.path.exists(dest):
        dest = os.path.join(input_dir, f"{stem}_{k}{ext}")
        k += 1
    shutil.copy(src_path, dest)

def ingest_zip(local_zip_path):
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(local_zip_path, "r") as zf:
            zf.extractall(tmp)
        for root, _, names in os.walk(tmp):
            if "__MACOSX" in root:
                continue
            for n in sorted(names):
                if is_image(n):
                    _copy_into_input(os.path.join(root, n), n)

def ingest_pdf(local_pdf_path):
    doc = fitz.open(local_pdf_path)
    base = os.path.splitext(os.path.basename(local_pdf_path))[0]
    mat = fitz.Matrix(PDF_DPI / 72, PDF_DPI / 72)
    for i, page in enumerate(doc):
        page.get_pixmap(matrix=mat).save(os.path.join(input_dir, f"{base}_p{i+1:03d}.png"))
    print(f"  {local_pdf_path}: {len(doc)} page(s) rasterised at {PDF_DPI} DPI")

def ingest_images_folder(local_folder):
    for fname in sorted(os.listdir(local_folder)):
        if is_image(fname):
            _copy_into_input(os.path.join(local_folder, fname), fname)

def ingest_any_local(local_path):
    if local_path.lower().endswith(".zip"):
        ingest_zip(local_path)
    elif local_path.lower().endswith(".pdf"):
        ingest_pdf(local_path)
    elif os.path.isdir(local_path):
        ingest_images_folder(local_path)
    elif is_image(local_path):
        _copy_into_input(local_path, os.path.basename(local_path))
    else:
        raise ValueError(f"Don't know how to handle: {local_path} (expected .zip, .pdf, an image, or a folder)")


if UPLOAD_INSTEAD:
    from google.colab import files
    print("Select a zip, a PDF, or one/several images...")
    uploaded = files.upload()
    for fname in uploaded:
        ingest_any_local(fname)

elif STORAGE_BACKEND == "drive":
    ingest_any_local(SOURCE_PATH)

else:  # onedrive
    if ONEDRIVE_SOURCE_PATH.lower().endswith((".zip", ".pdf")):
        local_tmp = "/content/" + os.path.basename(ONEDRIVE_SOURCE_PATH)
        onedrive_download_file(ONEDRIVE_SOURCE_PATH, local_tmp)
        ingest_any_local(local_tmp)
    elif is_image(ONEDRIVE_SOURCE_PATH):
        local_tmp = "/content/" + os.path.basename(ONEDRIVE_SOURCE_PATH)
        onedrive_download_file(ONEDRIVE_SOURCE_PATH, local_tmp)
        ingest_any_local(local_tmp)
    else:
        local_tmp = "/content/onedrive_source_images"
        onedrive_download_folder(ONEDRIVE_SOURCE_PATH, local_tmp)
        ingest_images_folder(local_tmp)

n_found = len([f for f in os.listdir(input_dir) if is_image(f)])
print(f"\nReady: {n_found} image(s) in {input_dir}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 40.3 MB/s eta 0:00:00
Select a zip, a PDF, or one/several images...


Saving C2-ThaliaMap.jpg to C2-ThaliaMap.jpg

Ready: 1 image(s) in /content/input_images


## Step 2 — Resize and enhance
---
Resizes by `resize_percent`, applies contrast per the config, then saves each page under the
size cap (JPEGs by lowering quality; PNG/WEBP by shrinking dimensions).

In [4]:
from PIL import Image, ImageEnhance
from tqdm import tqdm


def save_under_limit(img, path, max_bytes):
    ext = os.path.splitext(path)[1].lower()

    if ext in (".jpg", ".jpeg"):
        if img.mode not in ("RGB", "L"):
            img = img.convert("RGB")   # JPEG can't store alpha/palette modes
        quality = 95
        while quality >= 10:
            img.save(path, quality=quality, optimize=True)
            if os.path.getsize(path) <= max_bytes:
                return True
            quality -= 5
        return False  # couldn't hit target even at quality=10

    else:
        # PNG/WEBP/etc have no simple quality knob for size,
        # so keep shrinking dimensions instead
        current = img
        while True:
            current.save(path, optimize=True)
            if os.path.getsize(path) <= max_bytes:
                return True
            w, h = current.size
            if w < 100 or h < 100:  # safety floor
                return False
            current = current.resize((int(w * 0.9), int(h * 0.9)), Image.LANCZOS)


def enhance_image(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)


for filename in tqdm(sorted(os.listdir(input_dir))):
    if is_image(filename):
        in_path = os.path.join(input_dir, filename)
        out_path = os.path.join(output_dir, filename)

        with Image.open(in_path) as im:
            img = im.copy()

        # Resize by percentage
        new_w = int(img.width * resize_percent / 100)
        new_h = int(img.height * resize_percent / 100)
        img = img.resize((new_w, new_h), Image.LANCZOS)

        # Contrast enhancement (toggleable)
        if enhance_contrast:
            img = enhance_image(img, contrast_factor)

        success = save_under_limit(img, out_path, max_size_bytes)

        status = "OK" if success else "COULD NOT HIT TARGET"
        print(f"{filename}: {img.size[0]}x{img.size[1]} -> {os.path.getsize(out_path)/1024/1024:.2f} MB [{status}]")

print("Done")

100%|██████████| 1/1 [00:00<00:00,  4.58it/s]

C2-ThaliaMap.jpg: 1691x1536 -> 0.29 MB [OK]
Done


## Step 3 — Save the results
---
Saves to Drive / OneDrive per `STORAGE_BACKEND`. With `STORAGE_BACKEND = "none"` (or
`DOWNLOAD_ZIP = True`) it also bundles everything into a zip and downloads it to your computer.

In [5]:
if STORAGE_BACKEND in ("drive", "onedrive"):
    saved_location = save_to_storage(output_dir, save_name)

if STORAGE_BACKEND == "none" or DOWNLOAD_ZIP:
    from google.colab import files
    zip_base = f"/content/{save_name}"
    zip_file = shutil.make_archive(zip_base, "zip", output_dir)
    print(f"Zipped results: {zip_file} ({os.path.getsize(zip_file)/1024/1024:.1f} MB) — downloading...")
    files.download(zip_file)

Zipped results: /content/AntonioPiaggio_prepped_photos.zip (0.3 MB) — downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>